# Mi medida de la fiebre de GTA VI

**Roberto Cantero · Nebula · 23 de noviembre de 2026**

## Mi pregunta

Quiero anticipar las consultas de la página de GTA VI en Wikipedia en inglés durante las semanas tercera y quinta tras su lanzamiento. Mido consultas, no lectores únicos, jugadores ni ventas. Separo esta preparación de mi modelo y su evaluación. Uso días UTC y conservo todos los datos necesarios para repetir el trabajo sin conexión.

Necesito Python 3.10+ y NumPy. Ejecuto los cuadernos en esta carpeta, con modelo.py y los JSON al lado. Las salidas incluidas se han ejecutado en orden mediante Python, sin un servidor de Jupyter.

In [1]:
# leo mis datos conservados y genero el csv sin imputar ausencias
from modelo import preparar, RAIZ, EXCLUSIONES
from collections import Counter
import hashlib, json
import numpy as np
juegos, gta = preparar()
print('NumPy:', np.__version__)
print('Juegos conservados: 30; juegos utilizados:', len(juegos))
print('Mi reparto:', dict(Counter(s['grupo'] for s in juegos)))
print('Mis exclusiones:', EXCLUSIONES)
for nombre in ['series_consultadas.json','gta_vi_consultado.json']:
    print(nombre, hashlib.sha256((RAIZ/nombre).read_bytes()).hexdigest())

NumPy: 2.3.5
Juegos conservados: 30; juegos utilizados: 28
Mi reparto: {'entrenamiento': 14, 'validacion': 7, 'prueba': 7}
Mis exclusiones: {'Dark_Souls_III': 'estreno japonés anterior al mundial', "Baldur's_Gate_3": 'cobertura incompleta del título en los primeros días'}
series_consultadas.json 2987953f6c49fe831613f92af8e3bbcb2f4d5815d959fea7482e9317983a1cfc
gta_vi_consultado.json cdbd07d5e8dce7b652d835e1ebe9cb7b56582428e617daa98501778b12381105


## Mi procedencia y mis decisiones

He extraído las series del JSON visible del servicio oficial de Wikimedia, con proyecto en.wikipedia.org, all-access, user y frecuencia diaria. Conservo los números extraídos, no afirmo haber conservado el cuerpo bruto completo de cada respuesta. En series_consultadas.json guardo título, fecha de estreno y 43 consultas diarias consecutivas desde siete días antes hasta 35 días después. En fuentes_metodologia.md dejo las peticiones exactas y las páginas consultadas para las fechas.

Elijo una muestra de conveniencia de grandes videojuegos de pago de 2015 a 2023; no es representativa de todos los videojuegos. Conservo treinta series y excluyo dos por criterios de comparabilidad y cobertura. No incluyo GTA V: su estreno de 2013 precede a la cobertura diaria de este servicio. Dark Souls III tuvo un estreno japonés anterior al mundial; el título consultado de Baldur’s Gate 3 muestra consultas casi nulas al principio y un salto posterior, compatible con un cambio de título o redirección. No lo interpreto como ausencia de interés.

In [2]:
# compruebo la integridad de mis ventanas y las fechas de cada día
import csv
from datetime import date, timedelta
filas = list(csv.DictReader((RAIZ/'consultas_diarias.csv').open(encoding='utf-8')))
assert len(filas)==30*43
assert len({(s['titulo'],s['fecha']) for s in filas})==len(filas)
assert all(int(s['consultas'])>0 for s in filas)
assert len(gta['v'])==22
assert date.fromisoformat(gta['start'])+timedelta(days=21)==date(2026,11,22)
print('Filas conservadas:',len(filas))
print('Consultas de GTA VI del 19 al 22 de noviembre:',gta['v'][18:22])
print('Mis objetivos usan días +14 a +20 y +28 a +34; mis entradas solo -7 a +3.')

Filas conservadas: 1290
Consultas de GTA VI del 19 al 22 de noviembre: [750227, 540793, 393338, 308867]
Mis objetivos usan días +14 a +20 y +28 a +34; mis entradas solo -7 a +3.


## Mis límites antes de modelar

Una consulta no equivale a una persona. El idioma inglés no representa un país. El filtro user excluye agentes automatizados identificados, sin garantizar ausencia de toda automatización. Consultar un solo título no suma automáticamente las visitas de sus redirecciones históricas. Las fechas de estreno regionales y las nuevas versiones pueden afectar las curvas; no interpreto las diferencias como efectos causales de la calidad del juego. No convierto una caída de consultas en abandono del juego. Continúo en 02_prediccion.ipynb.